# Ascend Host offloading

Run native acceptance and inspect the fixed-capacity off/on matrix using reusable Python modules. Read ../README.md first. Select available chips; runs can take several minutes.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "experiments/homework3/run.py").exists()
)
SMALL_MODEL = ROOT / "outputs/models/Qwen3.5-0.8B"
LARGE_MODEL = ROOT / "outputs/models/Qwen3.5-4B"
DEVICES = "6,7"
TASK = "all"
OUTPUT = ROOT / "outputs/homework3/notebook-run"
RUN_NATIVE = False  # Set explicitly after selecting available chips.

In [ ]:
if RUN_NATIVE:
    env = os.environ.copy()
    paths = [
        ROOT,
        ROOT / "python",
        ROOT / "tokenspeed-kernel/python",
        ROOT / "tokenspeed-kernel-npu/python",
    ]
    env["PYTHONPATH"] = (
        os.pathsep.join(map(str, paths)) + os.pathsep + env.get("PYTHONPATH", "")
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "experiments.homework3.suite",
            "--small-model",
            str(SMALL_MODEL),
            "--large-model",
            str(LARGE_MODEL),
            "--devices",
            DEVICES,
            "--task",
            TASK,
            "--output",
            str(OUTPUT),
        ],
        cwd=ROOT,
        env=env,
        check=True,
    )

## Inspect measured evidence

Choose a completed matrix. `hits` are prefix hits; actual H2D bytes independently demonstrate Host restoration. End-to-end timing includes recomputation. Compare only identical Device budgets.


In [ ]:
MATRIX = ROOT / "outputs/homework3/performance_matrix.json"
matrix = json.loads(MATRIX.read_text())
assert matrix["passed"]
rows = []
for model, entry in matrix["models"].items():
    for run in entry["rows"]:
        for pattern, stats in run["workloads"].items():
            rows.append(
                {
                    "model": Path(model).name,
                    "host_ratio": run["host_ratio"],
                    "pattern": pattern,
                    **stats,
                }
            )
rows

For per-rank block counts, byte verification, transfer timings, and exact numerical checks, inspect each run's `analysis.json`, `rank*/transfers.jsonl`, and `continuation_comparison.json`. Incomplete or failed native cases must be fixed before interpreting a performance summary.
